# Set Up For Private Gemma 2 Model

1. Install bitsadnbytes to shrink the model to fit.


In [1]:
# Run this in a standard code cell
!pip install -q -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00


(Optional) Verify Functionality

In [ ]:
# Optional Code Block
import bitsandbytes as bnb
print(bnb.__version__)

0.49.2


2. Initialise the model

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from google.colab import userdata

# 1. Configuration
# We use the pre-quantized version to stay within the T4 GPU's 15GB VRAM
model_id = "unsloth/gemma-2-9b-it-bnb-4bit"

# Make sure your Colab Secret matches this name exactly (HF_TOKEN or HF-TOKEN)
token = userdata.get('HF-TOKEN')

# 2. Load Tokenizer and Model
# This will download to Colab's temporary local storage
print("Downloading model to temporary session... (Usually takes 2-3 minutes)")
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    token=token
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",          # Automatically maps to the T4 GPU
    torch_dtype=torch.bfloat16, # Optimized for Gemma 2
    token=token
)

print("Setup Complete! Model is ready in temporary memory.")

# 3. Test it out with the proper chat template
messages = [
    {
        "role": "user",
     "content": "Explain the concept of quantum entanglement like I'm five."
     }
]

# Gemma 2 requires this template to understand user vs. assistant roles
prompt = tokenizer.apply_chat_template(messages, tokenize=False,
                                       add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# 4. Generate with Memory Safety (torch.no_grad)
print("\nGenerating response...\n")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode and print the result
response = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):],
                            skip_special_tokens=True)
print(f"Gemma 2 says: \n{'-'*20}\n{response}")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/6.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Setup Complete! Model is ready in temporary memory.

Generating response...

Gemma 2 says: 
--------------------
Imagine you have two magic coins.  When you flip one, it always lands on heads, and when you flip the other, it always lands on tails. 

But here's the magic part: even if you put these coins far, far apart, like in different countries, if you flip one and see heads, you instantly know the other coin landed on tails! It's like they're connected by an invisible string, even when they're far away.

That's kind of like quantum entanglement! Tiny particles can be "linked" together, so that even when they're separated, they still act like they're connected. If you know something about one particle, you instantly know something about the other, even if they're miles apart!

It's very strange, even for grown-ups, but it's one of the coolest things about the tiny world of quantum mechanics!



3. Create a promting function

In [ ]:
def ask_gemma(question):
    # 1. Clear cache to make room for the "thought process"
    torch.cuda.empty_cache()

    # 2. Format the message
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False,
                                           add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # 3. Generate with the safety flags
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=True,
            temperature=0.7,
            use_cache=True,
            # Adding a pad token ID helps avoid some common warning errors
            pad_token_id=tokenizer.eos_token_id
        )

    # 4. Decode and return
    answer = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):],
                              skip_special_tokens=True)
    return answer

(Optional) Clear Cache Incase Memory for the context window is insufficient

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

4. Ask Questions Using the function

In [8]:
print(ask_gemma("How can I use colab to show me the chain of prompts I used on my model"))

You can't directly see the full chain of prompts you've used with a language model within Google Colab itself. Here's why and some potential workarounds:

**Why You Can't Directly View the Prompt History:**

* **Ephemeral Nature of Colab Sessions:** Colab notebooks are designed for interactive coding and experimentation. They don't inherently store a persistent history of all prompts and responses unless you explicitly save them. 
* **Model State Management:**  Language models often have a hidden state that's updated with each prompt. This state isn't directly accessible to you as a user. 

**Workarounds to Track Prompts:**

1. **Manual Logging:**

   * **Print Statements:** The most straightforward method is to use `print()` statements within your Colab code to record both the prompt you're sending to the model and the model's response.

     ```python
     prompt = "What is the capital of France?"
     response = model.generate(prompt) 
     print("Prompt:", prompt)
     print("Respo